In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys, zipfile

INPUT_ROOT = Path('/kaggle/input')
print('Kaggle input roots:', [p.name for p in INPUT_ROOT.iterdir()])
matches = list(INPUT_ROOT.rglob('reader.cp'))
assert len(matches) == 1, f'Expected one reader checkpoint in mounted inputs; found {matches}'
DATA = matches[0].parent
print('Using input:', DATA)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.3', 'hydra-core==1.3.2', 'jsonlines==4.0.0'])
for module, package in [('faiss', 'faiss-cpu'), ('spacy', 'spacy')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import torch, transformers, faiss, spacy
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('FAISS:', faiss.__version__)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Select Kaggle GPU T4 x2'


In [ ]:
import shutil

PROJECT_SOURCE = DATA / 'project'
PROJECT = Path('/kaggle/working/dpr-project')
shutil.copytree(PROJECT_SOURCE, PROJECT, dirs_exist_ok=True)
hf_path = PROJECT / 'dpr' / 'models' / 'hf_models.py'
hf_text = hf_path.read_text(encoding='utf-8')
legacy_padding_args = hf_text.count('pad_to_max_length=False')
assert legacy_padding_args == 2, f'Expected two old tokenizer padding arguments; found {legacy_padding_args}'
hf_path.write_text(hf_text.replace('pad_to_max_length=False', 'padding=False'), encoding='utf-8')
assert (PROJECT / 'scripts' / 'run_dpr_reader_reranking.py').is_file()
print('Working project:', PROJECT, '| tokenizer compatibility substitutions:', legacy_padding_args)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(PROJECT / 'scripts' / 'run_dpr_reader_reranking.py'),
    '--reader-checkpoint', str(DATA / 'reader.cp'),
    '--retriever-checkpoint', str(DATA / 'retriever.cp'),
    '--passage-vectors', str(DATA / 'passage_vectors.pkl'),
    '--corpus', str(DATA / 'wiki50k.tsv'),
    '--qa-dir', str(DATA),
    '--datasets', 'trivia_test',
    '--reader-gpus', '2',
    '--reader-batch-size', '16',
    '--top-docs', '200',
    '--output', '/kaggle/working/reranked_trivia_top200.json',
]
subprocess.run(cmd, check=True)
